In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import NMF, PCA
from scipy.sparse import hstack

In [2]:
df = pd.read_csv('E:/Works/10. Mental Health Disorder/Dataset/2_types_filtered_features_3lakh.csv')
df.head()

,text,Label,filtered_text_eng,tokenized_text,Preprocessed_Text,filtered_tokenized_words
0,"I'm done with it all. Any tips?First of all, i...",2,done with it Any of if going to comment get or...,"['done', 'go', 'comment', 'get', 'plea', 'unde...",done go comment get plea understand want hear ...,"['kill', 'pain', 'sorri', 'hate']"
1,i 20m was a problem child when grow up i frequ...,1,i was a problem child when grow up i frequent ...,"['problem', 'child', 'grow', 'frequent', 'got'...",problem child grow frequent got troubl junior ...,"['ill', 'awkward', 'steal', 'silenc', 'wrong',..."
2,I officially hate my school We have to start p...,0,I officially hate my school We have to start o...,"['offici', 'hate', 'school', 'start', 'enter',...",offici hate school start enter caus fix,"['offici', 'hate', 'school', 'start', 'enter',..."
3,but onc again depress love to take that from m...,1,but onc again depress love to take that from m...,"['onc', 'depress', 'love', 'take', 'angri', 'w...",onc depress love take angri want cri wont come...,"['cri', 'angri', 'damn', 'tire', 'depress']"
4,"Starting today, I'm going to attempt to fix my...",0,Starting going to attempt to fix my sleep sche...,"['start', 'go', 'attempt', 'fix', 'sleep', 'sc...",start go attempt fix sleep schedul school coup...,"['start', 'go', 'attempt', 'fix', 'sleep', 'sc..."


In [3]:
# Convert tokenized words into plain text
# Function to fix the tokenized words and convert to plain text
df['Filtered_Preprocessed_Text'] = df['filtered_tokenized_words'].apply(
    lambda x: ' '.join([''.join(word) for word in x])
)
df.head()

,text,Label,filtered_text_eng,tokenized_text,Preprocessed_Text,filtered_tokenized_words,Filtered_Preprocessed_Text
0,"I'm done with it all. Any tips?First of all, i...",2,done with it Any of if going to comment get or...,"['done', 'go', 'comment', 'get', 'plea', 'unde...",done go comment get plea understand want hear ...,"['kill', 'pain', 'sorri', 'hate']","[ ' k i l l ' , ' p a i n ' , ' s o r r i ..."
1,i 20m was a problem child when grow up i frequ...,1,i was a problem child when grow up i frequent ...,"['problem', 'child', 'grow', 'frequent', 'got'...",problem child grow frequent got troubl junior ...,"['ill', 'awkward', 'steal', 'silenc', 'wrong',...","[ ' i l l ' , ' a w k w a r d ' , ' s t e ..."
2,I officially hate my school We have to start p...,0,I officially hate my school We have to start o...,"['offici', 'hate', 'school', 'start', 'enter',...",offici hate school start enter caus fix,"['offici', 'hate', 'school', 'start', 'enter',...","[ ' o f f i c i ' , ' h a t e ' , ' s c h ..."
3,but onc again depress love to take that from m...,1,but onc again depress love to take that from m...,"['onc', 'depress', 'love', 'take', 'angri', 'w...",onc depress love take angri want cri wont come...,"['cri', 'angri', 'damn', 'tire', 'depress']","[ ' c r i ' , ' a n g r i ' , ' d a m n ' ..."
4,"Starting today, I'm going to attempt to fix my...",0,Starting going to attempt to fix my sleep sche...,"['start', 'go', 'attempt', 'fix', 'sleep', 'sc...",start go attempt fix sleep schedul school coup...,"['start', 'go', 'attempt', 'fix', 'sleep', 'sc...","[ ' s t a r t ' , ' g o ' , ' a t t e m p ..."


In [4]:
# Resulting DataFrame
print(df[['Filtered_Preprocessed_Text']])

                               Filtered_Preprocessed_Text
0       [ ' k i l l ' ,   ' p a i n ' ,   ' s o r r i ...
1       [ ' i l l ' ,   ' a w k w a r d ' ,   ' s t e ...
2       [ ' o f f i c i ' ,   ' h a t e ' ,   ' s c h ...
3       [ ' c r i ' ,   ' a n g r i ' ,   ' d a m n ' ...
4       [ ' s t a r t ' ,   ' g o ' ,   ' a t t e m p ...
...                                                   ...
287393  [ ' c r i ' ,   ' k n o w ' ,   ' c l a s s ' ...
287394  [ ' h a r d ' ,   ' i l l ' ,   ' s e l f i s ...
287395  [ ' a n g r i ' ,   ' l i k e ' ,   ' b u r n ...
287396  [ ' k i l l ' ,   ' s e v e r ' ,   ' h a r d ...
287397  [ ' r u i n ' ,   ' i s o l ' ,   ' w o r s t ...

[287398 rows x 1 columns]


In [5]:
df.isnull().sum()

text                          0
Label                         0
filtered_text_eng             0
tokenized_text                0
Preprocessed_Text             0
filtered_tokenized_words      0
Filtered_Preprocessed_Text    0
dtype: int64

In [6]:
df.shape

(287398, 7)

In [7]:
# BERT
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased")

In [8]:
# ELECTRA
electra_tokenizer = AutoTokenizer.from_pretrained("google/electra-small-discriminator")
electra_model = AutoModel.from_pretrained("google/electra-small-discriminator")

In [9]:
# Initialize GPT-2 tokenizer and model
gpt_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt_model = AutoModel.from_pretrained("gpt2")

# Set pad token as eos token for GPT-2
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

In [10]:
# XLNet
xlnet_tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased")
xlnet_model = AutoModel.from_pretrained("xlnet-base-cased")

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Name: NVIDIA GeForce RTX 4070 Laptop GPU


In [12]:
# Move models to GPU
bert_model.to(device)
electra_model.to(device)
gpt_model.to(device)
xlnet_model.to(device)

XLNetModel(
  (word_embedding): Embedding(32000, 768)
  (layer): ModuleList(
    (0-11): 12 x XLNetLayer(
      (rel_attn): XLNetRelativeAttention(
        (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): XLNetFeedForward(
        (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (layer_1): Linear(in_features=768, out_features=3072, bias=True)
        (layer_2): Linear(in_features=3072, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (activation_function): GELUActivation()
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
)

In [13]:
def get_transformer_embeddings(texts, tokenizer, model, device=device, batch_size=16):
    embeddings = []
    model.to(device)  # Move model to device (CPU or GPU)
    model.eval()  # Set model to evaluation mode

    with torch.no_grad():
        # Process texts in batches
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(batch_texts, return_tensors='pt', truncation=True, padding=True, max_length=128)
            input_ids = inputs['input_ids'].to(device)
            attention_mask = inputs['attention_mask'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            last_hidden_states = outputs.last_hidden_state
            batch_embeddings = last_hidden_states.mean(dim=1).cpu().numpy()
            embeddings.append(batch_embeddings)

    return np.vstack(embeddings)

In [14]:
X_bert = get_transformer_embeddings(df['Preprocessed_Text'].tolist(), bert_tokenizer, bert_model, device=device)

In [15]:
X_electra = get_transformer_embeddings(df['Preprocessed_Text'].tolist(), electra_tokenizer, electra_model, device=device)

In [16]:
X_gpt = get_transformer_embeddings(df['Preprocessed_Text'].tolist(), gpt_tokenizer, gpt_model, device=device)

In [17]:
X_xlnet = get_transformer_embeddings(df['Preprocessed_Text'].tolist(), xlnet_tokenizer, xlnet_model, device=device)

In [20]:
# Load GloVe Embeddings
glove_embeddings = {}
glove_file_path = "E:/Works/10. Mental Health Disorder/Dataset/Extra Documents/glove.6B.100d.txt"
with open(glove_file_path, encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        glove_embeddings[word] = vector

# Word Embeddings using GloVe
def get_glove_vector(tokens, glove_embeddings, dim=100):
    embeddings = [glove_embeddings.get(word, np.zeros(dim)) for word in tokens]
    return np.mean(embeddings, axis=0) if embeddings else np.zeros(dim)

In [21]:
X_glove = np.array([get_glove_vector(tokens, glove_embeddings) for tokens in df["tokenized_text"]])

In [22]:
# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=100, ngram_range=(1, 3))
X_tfidf = tfidf_vectorizer.fit_transform(df["tokenized_text"]).toarray()

In [23]:
from sklearn.feature_extraction.text import CountVectorizer

# Count Vectorizer with custom tokenizer and preprocessor
count_vectorizer = CountVectorizer(
    max_features=100,           # Limit to 100 features
    tokenizer=lambda x: x,      
    preprocessor=lambda x: x   
)

# Transform the data
X_count = count_vectorizer.fit_transform(df['tokenized_text']).toarray()

E:\Works\10. Mental Health Disorder\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [24]:
X_combined = np.hstack((X_glove, X_tfidf, X_count, X_bert, X_electra, X_gpt, X_xlnet))

In [25]:
# Convert combined features to tensor
X_final = torch.tensor(X_combined, dtype=torch.float32)

In [27]:
# Convert tensor to NumPy array
X_final_np = X_final.numpy()

# Create a DataFrame from the NumPy array
df_final = pd.DataFrame(X_final_np)

# Save the DataFrame to a CSV file 
df_final.to_csv('E:/Works/10. Mental Health Disorder/Dataset/2_types_all_features.csv', index=False)

In [28]:
df_final.head(20)

,0,1,2,3,4,5,6,7,8,9,...,2781,2782,2783,2784,2785,2786,2787,2788,2789,2790
0,-0.303352,0.142832,0.257117,-0.347635,-0.334661,-0.001531,0.118400,0.050449,-0.288898,-0.011938,...,1.174970,-0.131031,0.581714,1.317830,0.545295,-4.912286,2.002300,-2.429307,2.313618,0.068319
1,-0.302524,0.168964,0.247727,-0.321375,-0.378240,-0.012802,0.129170,0.041635,-0.320993,-0.033437,...,-0.170022,-1.057126,0.506789,0.481091,-0.443754,-1.015675,0.295429,-1.521313,-0.511026,-0.270724
2,-0.258834,0.199409,0.255281,-0.329083,-0.421285,-0.008494,0.122459,0.000529,-0.345936,-0.007731,...,-0.475027,-2.049421,-2.095691,-2.270160,2.747773,-4.004610,-1.818188,-3.040984,-0.021312,1.718756
3,-0.331483,0.133704,0.255817,-0.369843,-0.361175,0.004250,0.104048,0.029749,-0.330116,-0.050912,...,1.116824,0.496518,2.048387,-0.856290,0.582637,-1.573216,0.650256,-1.483604,0.080413,-0.893228
4,-0.258569,0.201524,0.242732,-0.321395,-0.432100,-0.044947,0.111697,0.009705,-0.350573,-0.033735,...,-0.861711,-1.552959,-0.070510,-2.612565,0.754691,-5.594378,3.663499,-0.440780,1.437583,-0.616007
5,-0.291338,0.148380,0.240409,-0.317561,-0.360210,-0.014240,0.114810,0.025244,-0.319628,-0.019425,...,0.328832,-0.130062,-0.398639,0.483432,0.527055,-2.224774,0.269343,-2.088396,1.147085,-1.117991
6,-0.331597,0.171952,0.254036,-0.326541,-0.400527,-0.014695,0.125140,0.033078,-0.337841,-0.033034,...,-0.195479,-1.666451,0.691748,0.216133,0.736730,-3.127230,-0.526722,-1.756144,-0.267478,-0.249440
7,-0.316708,0.169958,0.213350,-0.366424,-0.353601,0.015562,0.124488,-0.027506,-0.358802,-0.046981,...,1.368073,-1.310364,-2.792416,-4.041128,0.888530,-6.746810,-0.641369,0.891115,2.391107,-0.535521
8,-0.264632,0.172277,0.244350,-0.331715,-0.430998,-0.010690,0.115765,0.030654,-0.314289,-0.062362,...,1.064479,-0.440749,0.247903,-1.815316,0.336433,-4.959431,1.330520,-3.148598,-0.100644,0.013222
9,-0.281939,0.166514,0.235172,-0.350586,-0.360470,-0.022592,0.132992,0.024310,-0.314069,-0.023638,...,1.688184,-1.166411,1.806010,0.238370,-1.340917,-3.654134,2.615348,-1.513550,2.204821,-1.370534


In [29]:
# Convert combined features to tensor
X_bert_final = torch.tensor(X_bert, dtype=torch.float32)

# Convert tensor to NumPy array
X_bert_final_np = X_bert_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_bert = pd.DataFrame(X_bert_final_np)

# Save the DataFrame to a CSV file 
df_final_X_bert.to_csv('E:/Works/10. Mental Health Disorder/Dataset/2_types_bert.csv', index=False)
df_final_X_bert.head(20)

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,0.039339,0.027318,0.542960,0.067278,0.035454,0.057449,0.192008,0.011376,0.099094,-0.457100,...,-0.026903,-0.287609,-0.026079,-0.081054,-0.172036,0.204499,-0.023147,-0.176354,-0.130857,0.055808
1,-0.491383,-0.116573,0.474932,-0.000259,0.101972,-0.092985,0.298460,0.312335,0.049858,-0.411412,...,-0.029965,-0.137378,0.017629,0.033461,-0.035370,0.011849,-0.204691,-0.353505,-0.099069,0.056156
2,0.004747,-0.169299,0.094293,-0.033749,0.114623,0.008532,0.428237,0.353712,0.106186,-0.518745,...,0.207332,-0.199377,0.141076,0.200267,-0.090277,-0.081951,0.014670,-0.250805,-0.021329,0.226560
3,-0.203363,-0.051969,0.642216,-0.132124,-0.112664,-0.007807,0.282154,0.017729,0.194809,-0.252398,...,-0.049852,-0.410158,-0.010218,0.092821,-0.256525,0.253892,-0.082146,-0.270716,-0.134515,0.038670
4,-0.062877,-0.319566,0.300076,0.008816,0.305639,-0.206012,0.217093,-0.085044,0.095859,-0.175441,...,0.122657,-0.236463,0.093678,-0.079467,-0.015312,0.244647,-0.044545,-0.148861,-0.062770,-0.102613
5,-0.146716,0.028845,0.487284,0.010830,-0.097013,-0.161656,0.276186,0.372388,0.296024,-0.335906,...,0.313471,-0.226476,-0.075263,0.129500,-0.141699,0.119047,-0.013919,-0.285261,-0.007778,0.051887
6,-0.229220,-0.071243,0.659668,0.057474,0.180020,-0.169152,0.083694,0.090853,0.136538,-0.457813,...,0.131410,-0.367415,0.155208,-0.029633,0.013474,0.161375,-0.066659,-0.361590,-0.200353,0.137712
7,-0.024275,0.140617,0.210344,-0.032038,-0.129329,0.106952,0.298608,0.525600,0.201474,-0.591594,...,0.044255,-0.403285,0.117265,0.106770,-0.120464,-0.056509,0.219332,0.027430,0.399939,-0.029759
8,0.101469,-0.151218,0.309608,0.149115,0.133245,-0.149087,0.254839,0.013539,0.158101,-0.155652,...,0.188079,-0.379465,0.075879,-0.038100,0.070519,0.288737,-0.027767,-0.299966,-0.065827,0.114660
9,0.079475,-0.306717,0.467059,0.079831,0.261917,-0.038099,0.228091,0.080303,0.087497,-0.394976,...,0.145723,-0.326566,0.310158,-0.022558,-0.140809,0.093799,-0.190606,-0.129748,-0.178369,-0.067853


In [30]:
# Convert combined features to tensor
X_electra_final = torch.tensor(X_electra, dtype=torch.float32)

# Convert tensor to NumPy array
X_electra_final_np = X_electra_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_electra = pd.DataFrame(X_electra_final_np)

# Save the DataFrame to a CSV file 
df_final_X_electra.to_csv('E:/Works/10. Mental Health Disorder/Dataset/2_types_electra.csv', index=False)
df_final_X_electra.head(20)

,0,1,2,3,4,5,6,7,8,9,...,246,247,248,249,250,251,252,253,254,255
0,0.599472,0.439625,0.020266,0.138930,-0.609430,-0.539496,-0.403593,-0.137967,0.020127,0.690898,...,0.141068,0.108030,-0.510015,-1.147126,-0.002389,-0.206351,1.127947,-0.253208,-0.523452,-0.328024
1,0.250966,0.313902,-0.319763,0.109656,-0.361788,-0.077295,0.165057,0.165286,0.238636,1.128708,...,0.042352,0.410725,-0.239817,-0.738910,-0.009571,-0.778596,0.321679,-0.342413,-0.209100,-0.151962
2,0.798442,0.137798,0.217636,0.615689,-0.422624,-0.667775,-0.803210,-0.369767,-0.318563,-0.041989,...,0.041961,-0.172283,-0.642360,-0.670354,-0.119998,0.519885,1.531320,-0.087807,-0.414092,-0.394284
3,0.694335,0.538832,0.078782,0.329488,-0.570069,-0.392095,-0.639536,-0.064476,-0.135712,0.484850,...,0.131050,-0.011844,-0.597980,-0.633558,0.011596,0.206810,1.234858,-0.258696,-0.551979,-0.394734
4,0.753654,0.343877,0.177863,0.563892,-0.583629,-0.589238,-0.902104,-0.213953,-0.210492,0.123507,...,0.040913,-0.103077,-0.607364,-0.749816,0.032885,0.400657,1.501543,-0.132509,-0.436215,-0.385705
5,0.556395,0.336273,-0.044140,0.330744,-0.528796,-0.410875,-0.402967,-0.000822,-0.132016,0.692665,...,0.120789,0.038540,-0.543584,-0.826162,0.019727,-0.075651,1.121055,-0.201159,-0.669734,-0.347811
6,0.606831,0.416998,-0.089709,0.383874,-0.437087,-0.447979,-0.397252,0.010724,-0.038785,0.519137,...,0.102338,0.211360,-0.472772,-0.861665,-0.001111,-0.238651,0.985930,-0.143281,-0.557462,-0.206413
7,0.889869,0.200160,0.211763,0.602618,-0.216460,-0.731466,-0.723430,-0.237504,-0.379962,-0.128080,...,-0.015177,-0.200958,-0.557886,-0.615360,-0.241240,0.829682,1.098042,-0.295733,-0.741910,-0.611906
8,0.490454,0.138842,0.093108,0.606177,-0.387260,-0.540420,-0.358276,-0.192229,-0.303791,0.096348,...,0.186031,-0.042873,-0.751456,-0.775492,-0.183552,0.389557,1.154379,-0.125626,-0.509174,-0.514307
9,0.506974,0.091086,0.002914,0.419425,-0.413436,-0.423929,-0.367566,0.084806,-0.206073,0.511355,...,0.238478,0.067842,-0.698653,-0.871200,-0.124420,-0.003842,1.044603,-0.006546,-0.466472,-0.294304


In [31]:
# Convert combined features to tensor
X_gpt_final = torch.tensor(X_gpt, dtype=torch.float32)

# Convert tensor to NumPy array
X_gpt_final_np = X_gpt_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_gpt = pd.DataFrame(X_gpt_final_np)

# Save the DataFrame to a CSV file 
df_final_X_gpt.to_csv('E:/Works/10. Mental Health Disorder/Dataset/2_types_gpt.csv', index=False)
df_final_X_gpt.head(20)

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,-0.175703,-0.128528,-0.084935,-0.095990,0.017343,0.040334,4.020565,-0.036600,0.177849,0.013525,...,-0.195830,0.354231,-0.207117,-0.020549,2.571847,0.269671,0.355543,-0.200343,0.132587,0.053167
1,-0.053351,-0.158133,-0.083422,0.027374,-0.146508,0.105708,-1.912806,-0.095140,0.337503,0.320859,...,-0.540632,0.145271,-0.082381,0.016155,4.346670,0.138111,0.160342,-0.194967,0.396577,0.260527
2,-0.104494,-0.081889,0.118526,-0.011474,0.102028,-0.325892,4.923399,-0.233921,0.015988,0.042901,...,-0.197509,0.139917,-0.156645,-0.157115,0.840378,0.213294,0.154508,-0.011298,0.103850,-0.031862
3,-0.161296,-0.008129,-0.258956,-0.103768,0.145393,0.025114,13.907259,0.098031,-0.028736,-0.069874,...,-0.077221,0.282741,-0.155734,-0.002357,2.100598,0.299896,0.207297,-0.125741,-0.037304,-0.047917
4,-0.001621,-0.146236,0.058283,-0.089719,0.079217,-0.047845,10.313011,-0.054384,0.008916,-0.126622,...,-0.053217,0.129640,-0.228580,-0.135887,0.790294,0.242170,0.237762,-0.040008,-0.005287,-0.090784
5,-0.230178,0.059749,-0.237078,-0.141400,0.097137,0.059281,11.449934,0.014305,0.171883,0.031766,...,-0.110786,0.379071,-0.150733,0.045897,2.436573,0.209442,0.264724,-0.183146,0.070527,-0.093800
6,-0.150603,0.035567,-0.245175,-0.099362,0.055164,-0.067794,9.465213,0.052796,0.045801,0.070554,...,-0.065945,0.544943,-0.012305,-0.177044,3.173668,0.233340,0.136867,-0.149710,0.054362,0.020757
7,-0.245143,0.066866,-0.160786,-0.152481,0.051272,-0.041539,16.373730,0.082881,-0.049279,-0.152321,...,-0.017368,0.272946,-0.289343,-0.009536,1.389789,0.173426,0.198056,-0.010321,-0.249162,-0.197115
8,-0.084476,0.074708,-0.167517,-0.167617,0.092356,-0.011225,13.419597,0.023389,0.009952,0.063642,...,-0.079386,0.359535,-0.133965,-0.102594,1.909373,0.318847,0.237308,-0.090948,-0.061338,-0.134615
9,-0.312247,-0.128049,-0.151016,0.009313,0.109228,0.071397,7.738452,0.015650,0.046567,0.027857,...,-0.011443,0.334339,-0.153145,-0.099489,2.088678,0.279512,0.231282,0.004075,0.031337,-0.026658


In [32]:
# Convert combined features to tensor
X_xlnet_final = torch.tensor(X_xlnet, dtype=torch.float32)

# Convert tensor to NumPy array
X_xlnet_final_np = X_xlnet_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_xlnet = pd.DataFrame(X_xlnet_final_np)

# Save the DataFrame to a CSV file 
df_final_X_xlnet.to_csv('E:/Works/10. Mental Health Disorder/Dataset/2_types_xlnet.csv', index=False)
df_final_X_xlnet.head(20)

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,0.759141,-0.219632,-2.537091,1.348352,0.156356,-1.809287,0.046762,1.175762,-4.395438,0.086577,...,1.174970,-0.131031,0.581714,1.317830,0.545295,-4.912286,2.002300,-2.429307,2.313618,0.068319
1,-0.061250,-0.591830,-1.094051,0.190955,-0.056016,-0.131817,0.305451,0.940083,-0.292015,-0.022841,...,-0.170022,-1.057126,0.506789,0.481091,-0.443754,-1.015675,0.295429,-1.521313,-0.511026,-0.270724
2,1.201082,-2.194241,-0.574723,-1.052984,-3.301701,-5.273770,-0.233264,4.624631,0.466483,1.650347,...,-0.475027,-2.049421,-2.095691,-2.270160,2.747773,-4.004610,-1.818188,-3.040984,-0.021312,1.718756
3,-0.163426,0.753552,-1.646212,0.441858,-1.658332,0.302774,-0.637836,2.469987,-1.733064,0.607669,...,1.116824,0.496518,2.048387,-0.856290,0.582637,-1.573216,0.650256,-1.483604,0.080413,-0.893228
4,-0.700492,-0.445793,-1.847681,2.117335,-1.111673,-1.074618,-2.676463,1.812645,-1.588386,-0.335436,...,-0.861711,-1.552959,-0.070510,-2.612565,0.754691,-5.594378,3.663499,-0.440780,1.437583,-0.616007
5,-1.118945,-0.317040,-1.448066,-0.214862,-0.322028,-2.265767,-0.303574,2.556226,-2.614884,1.630911,...,0.328832,-0.130062,-0.398639,0.483432,0.527055,-2.224774,0.269343,-2.088396,1.147085,-1.117991
6,1.429431,-0.042020,-1.947453,0.460862,-2.092767,-2.315763,-1.036957,1.579913,-1.373635,0.517810,...,-0.195479,-1.666451,0.691748,0.216133,0.736730,-3.127230,-0.526722,-1.756144,-0.267478,-0.249440
7,1.196044,-0.799787,-1.495345,-3.110965,-1.556351,-4.769227,-1.586586,0.719480,-1.252639,3.661522,...,1.368073,-1.310364,-2.792416,-4.041128,0.888530,-6.746810,-0.641369,0.891115,2.391107,-0.535521
8,0.293644,-0.243832,-2.146861,1.271089,-0.764019,-0.741192,0.208977,1.083503,-2.964895,1.536098,...,1.064479,-0.440749,0.247903,-1.815316,0.336433,-4.959431,1.330520,-3.148598,-0.100644,0.013222
9,1.239199,-1.966398,-1.242368,-0.064255,-0.578584,-1.580598,-0.196727,1.028074,-2.087779,-1.607387,...,1.688184,-1.166411,1.806010,0.238370,-1.340917,-3.654134,2.615348,-1.513550,2.204821,-1.370534


In [33]:
# Convert combined features to tensor
X_glove_final = torch.tensor(X_glove, dtype=torch.float32)

# Convert tensor to NumPy array
X_glove_final_np = X_glove_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_glove = pd.DataFrame(X_glove_final_np)

# Save the DataFrame to a CSV file 
df_final_X_glove.to_csv('E:/Works/10. Mental Health Disorder/Dataset/2_types_glove.csv', index=False)
df_final_X_glove.head(20)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.303352,0.142832,0.257117,-0.347635,-0.334661,-0.001531,0.118400,0.050449,-0.288898,-0.011938,...,-0.294396,0.146451,0.203953,0.397330,-0.109241,-0.088820,-0.123558,-0.710045,0.431673,-0.072842
1,-0.302524,0.168964,0.247727,-0.321375,-0.378240,-0.012802,0.129170,0.041635,-0.320993,-0.033437,...,-0.338289,0.154216,0.233454,0.401414,-0.118155,-0.076909,-0.107612,-0.697643,0.393319,-0.094720
2,-0.258834,0.199409,0.255281,-0.329083,-0.421285,-0.008494,0.122459,0.000529,-0.345936,-0.007731,...,-0.265862,0.177597,0.220011,0.334313,-0.126213,-0.064588,-0.089246,-0.689013,0.371360,-0.098661
3,-0.331483,0.133704,0.255817,-0.369843,-0.361175,0.004250,0.104048,0.029749,-0.330116,-0.050912,...,-0.334936,0.140684,0.223424,0.403500,-0.135794,-0.084172,-0.099223,-0.712792,0.416418,-0.066589
4,-0.258569,0.201524,0.242732,-0.321395,-0.432100,-0.044947,0.111697,0.009705,-0.350573,-0.033735,...,-0.280412,0.195986,0.226042,0.345849,-0.083964,-0.068125,-0.107052,-0.717904,0.402261,-0.075529
5,-0.291338,0.148380,0.240409,-0.317561,-0.360210,-0.014240,0.114810,0.025244,-0.319628,-0.019425,...,-0.312488,0.172602,0.216087,0.407506,-0.114834,-0.086345,-0.096800,-0.687014,0.391150,-0.103078
6,-0.331597,0.171952,0.254036,-0.326541,-0.400527,-0.014695,0.125140,0.033078,-0.337841,-0.033034,...,-0.369696,0.174029,0.234615,0.425934,-0.107905,-0.074502,-0.093367,-0.697348,0.389353,-0.087435
7,-0.316708,0.169958,0.213350,-0.366424,-0.353601,0.015562,0.124488,-0.027506,-0.358802,-0.046981,...,-0.347679,0.206957,0.314258,0.413638,-0.072829,-0.022229,-0.079478,-0.720027,0.386465,-0.118947
8,-0.264632,0.172277,0.244350,-0.331715,-0.430998,-0.010690,0.115765,0.030654,-0.314289,-0.062362,...,-0.279743,0.140809,0.230463,0.349047,-0.116909,-0.083156,-0.128989,-0.691584,0.406910,-0.074826
9,-0.281939,0.166514,0.235172,-0.350586,-0.360470,-0.022592,0.132992,0.024310,-0.314069,-0.023638,...,-0.315188,0.177110,0.240630,0.409813,-0.102384,-0.043537,-0.118887,-0.713127,0.402605,-0.095379


In [34]:
# Convert combined features to tensor
X_tfidf_final = torch.tensor(X_tfidf, dtype=torch.float32)

# Convert tensor to NumPy array
X_tfidf_final_np = X_tfidf_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_tfidf = pd.DataFrame(X_tfidf_final_np)

# Save the DataFrame to a CSV file 
df_final_X_tfidf.to_csv('E:/Works/10. Mental Health Disorder/Dataset/2_types_tfidf.csv', index=False)
df_final_X_tfidf.head(20)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.000000,0.242895,0.000000,0.000000,0.0,0.000000,0.244922,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.449525,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.199899,0.078827,0.0,0.000000,0.000000,0.0,0.066124,0.000000,...,0.000000,0.152729,0.000000,0.07523,0.045237,0.064655,0.000000,0.000000,0.056853,0.064286
2,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.188929,0.000000,0.000000,0.00000,0.491190,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.114017,0.000000,0.000000,0.000000,0.000000,0.000000
7,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.323097,0.000000,...,0.000000,0.000000,0.270881,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [35]:
# Convert combined features to tensor
X_count_final = torch.tensor(X_count, dtype=torch.float32)

# Convert tensor to NumPy array
X_count_final_np = X_count_final.numpy()

# Create a DataFrame from the NumPy array
df_final_X_count = pd.DataFrame(X_count_final_np)

# Save the DataFrame to a CSV file 
df_final_X_count.to_csv('E:/Works/10. Mental Health Disorder/Dataset/2_types_count.csv', index=False)
df_final_X_count.head(20)

,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
0,47.0,96.0,47.0,1.0,1.0,29.0,1.0,4.0,7.0,28.0,...,0.0,16.0,16.0,17.0,5.0,3.0,5.0,0.0,3.0,0.0
1,234.0,470.0,234.0,1.0,1.0,75.0,19.0,44.0,48.0,136.0,...,4.0,70.0,49.0,96.0,47.0,19.0,28.0,2.0,16.0,2.0
2,6.0,14.0,6.0,1.0,1.0,3.0,0.0,3.0,0.0,3.0,...,0.0,2.0,3.0,4.0,1.0,0.0,0.0,1.0,0.0,0.0
3,27.0,56.0,27.0,1.0,1.0,11.0,0.0,3.0,5.0,17.0,...,0.0,8.0,6.0,12.0,0.0,2.0,6.0,0.0,0.0,0.0
4,13.0,28.0,13.0,1.0,1.0,3.0,0.0,3.0,1.0,9.0,...,0.0,3.0,7.0,9.0,2.0,0.0,0.0,1.0,0.0,0.0
5,41.0,84.0,41.0,1.0,1.0,16.0,2.0,9.0,9.0,26.0,...,0.0,13.0,11.0,22.0,11.0,3.0,0.0,0.0,0.0,0.0
6,45.0,92.0,45.0,1.0,1.0,11.0,1.0,6.0,12.0,31.0,...,0.0,12.0,9.0,14.0,8.0,2.0,5.0,0.0,0.0,0.0
7,6.0,14.0,6.0,1.0,1.0,3.0,1.0,2.0,1.0,7.0,...,0.0,3.0,2.0,2.0,0.0,1.0,2.0,0.0,0.0,0.0
8,22.0,46.0,22.0,1.0,1.0,6.0,1.0,3.0,1.0,13.0,...,1.0,5.0,5.0,14.0,3.0,3.0,1.0,0.0,0.0,0.0
9,37.0,76.0,37.0,1.0,1.0,15.0,4.0,12.0,10.0,27.0,...,0.0,10.0,11.0,9.0,5.0,1.0,1.0,0.0,2.0,0.0


In [36]:
print(df_final.shape)
print(df_final_X_bert.shape)
print(df_final_X_electra.shape)
print(df_final_X_gpt.shape)
print(df_final_X_xlnet.shape)
print(df_final_X_glove.shape)
print(df_final_X_tfidf.shape)
print(df_final_X_count.shape)

(287398, 2791)
(287398, 768)
(287398, 256)
(287398, 768)
(287398, 768)
(287398, 100)
(287398, 100)
(287398, 31)
